# SecureRAG Tutorial 3: Strict gRPC API

This notebook demonstrates using the strict method-per-operation gRPC service.

Before running, start the server in a terminal:

python -m securerag.grpc_server --host 127.0.0.1 --port 50051

In [ ]:
from securerag.backend_client import create_backend

In [ ]:
backend = create_backend("grpc://127.0.0.1:50051")

docs = [
    {"doc_id": "q3", "text": "Q3 risk report highlights vendor concentration and delayed remediation.", "metadata": {}},
    {"doc_id": "policy", "text": "Security policy requires quarterly risk treatment tracking and owner assignment.", "metadata": {}},
]

chunks = backend.chunk(docs=docs, chunk_size=120, overlap=32)
len(chunks), chunks[0].keys()

In [ ]:
key = backend.sse_generate_key()
prepared = backend.sse_prepare_chunks(
    chunks=chunks,
    key=key,
    scheme="structured",
    use_bigrams=True,
)
index = backend.build_index(protocol="EncryptedSearch", chunks=prepared)
index

In [ ]:
q_terms = backend.sse_encrypt_structured_terms(
    text="Summarize key risks in Q3",
    key=key,
    use_bigrams=True,
)
rows = backend.structured_search(index_id=index["index_id"], struct_terms=q_terms, top_k=3)
rows